# Exercises XP: LoRA (Low-Rank Adaptation) from Scratch in PyTorch

Week 13, Day 2

**LoRA idea.** Instead of updating a large weight matrix $W \in \mathbb{R}^{d_{out} \times d_{in}}$ during fine-tuning, we keep $W$ **frozen** and learn a low-rank update $\Delta W = \alpha \,(A B)^\top$ with $A \in \mathbb{R}^{d_{in} \times r}$ and $B \in \mathbb{R}^{r \times d_{out}}$, where the rank $r$ is small (e.g. 4). The layer computes

$$ y = x W^\top + b + \alpha \, x A B $$

The number of trainable parameters drops from $d_{in} \times d_{out}$ to $r \times (d_{in} + d_{out})$. $B$ is initialised to **zero**, so at the start $\Delta W = 0$ and the adapted model behaves exactly like the original one.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 🌟 Exercise 1: Implementing the LoRALayer

In [2]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)  # random init, scaled by 1/sqrt(rank)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))           # zero init -> the LoRA update starts at 0
        self.alpha = alpha                                          # scaling factor of the update

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)   # (batch, in_dim) @ (in_dim, rank) @ (rank, out_dim) -> (batch, out_dim)
        return x

In [3]:
torch.manual_seed(123)
lora_test = LoRALayer(in_dim=10, out_dim=2, rank=4, alpha=8)
x_small = torch.randn(1, 10)
print("A shape:", tuple(lora_test.A.shape), "| B shape:", tuple(lora_test.B.shape))
print("LoRA output with B = 0:", lora_test(x_small))      # all zeros at initialisation

# With a non-zero B the layer produces a real low-rank transformation
with torch.no_grad():
    lora_test.B.normal_()
print("LoRA output with random B:", lora_test(x_small))
print("Rank of the update A @ B:", torch.linalg.matrix_rank(lora_test.A @ lora_test.B).item(), "(at most the LoRA rank, here limited by out_dim=2)")

A shape: (10, 4) | B shape: (4, 2)
LoRA output with B = 0: tensor([[0., 0.]], grad_fn=<MulBackward0>)
LoRA output with random B: tensor([[-8.0818, 11.5079]], grad_fn=<MulBackward0>)
Rank of the update A @ B: 2 (at most the LoRA rank, here limited by out_dim=2)


The output has the expected shape `(1, 2)`. At initialisation it is exactly **zero** because `B` is all zeros: the LoRA branch does not change anything until it is trained. Once `B` is non-zero, the layer applies a transformation whose matrix `A @ B` has rank at most 4 (here 2, the output size).

## 🌟 Exercise 2: Implementing the LinearWithLoRA Layer

In [4]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear                       # the original (pretrained) linear layer
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)       # original output + low-rank adaptation

## 🌟 Exercise 3: Creating a Small Neural Network and Applying LoRA

In [5]:
# Hyperparameters
random_seed = 123

torch.manual_seed(random_seed)
layer = nn.Linear(10, 2)          # a single-layer network: 10 inputs -> 2 outputs
x = torch.randn((1, 10))          # random input tensor

print(x)
print(layer)
print('Original output:', layer(x))

tensor([[ 0.5490,  0.3671,  0.1219,  0.6466, -1.4168,  0.8429, -0.6307,  1.2340,
          0.3127,  0.6972]])
Linear(in_features=10, out_features=2, bias=True)
Original output: tensor([[0.6639, 0.4487]], grad_fn=<AddmmBackward0>)


In [6]:
## Applying LoRA to Linear Layer
layer_lora_1 = LinearWithLoRA(layer, rank=2, alpha=4)
print(layer_lora_1(x))
print("Identical to the original output:", torch.allclose(layer(x), layer_lora_1(x)))

tensor([[0.6639, 0.4487]], grad_fn=<AddBackward0>)
Identical to the original output: True


The `LinearWithLoRA` output is **exactly the same** as the original layer: since `B` starts at zero, the LoRA branch adds 0. Fine-tuning can then start from the pretrained behaviour and only learn a *change* to it.

## 🌟 Exercise 4: Merging LoRA Matrices and Testing Equivalence

In [7]:
# This LoRA code is equivalent to LinearWithLoRA
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        lora = self.lora.A @ self.lora.B  # combine LoRA matrices: (in_dim, out_dim)
        # then combine LoRA with the original weights (nn.Linear stores W as (out_dim, in_dim), hence the transpose)
        combined_weight = self.linear.weight + self.lora.alpha * lora.T
        return F.linear(x, combined_weight, self.linear.bias)

layer_lora_2 = LinearWithLoRAMerged(layer, rank=2, alpha=4)
print(layer_lora_2(x))

tensor([[0.6639, 0.4487]], grad_fn=<AddmmBackward0>)


In [8]:
# A stronger test: give both versions the same NON-zero LoRA matrices and compare the outputs
torch.manual_seed(0)
with torch.no_grad():
    layer_lora_1.lora.B.normal_()
    layer_lora_2.lora.A.copy_(layer_lora_1.lora.A)
    layer_lora_2.lora.B.copy_(layer_lora_1.lora.B)
out_1, out_2 = layer_lora_1(x), layer_lora_2(x)
print("LinearWithLoRA      :", out_1)
print("LinearWithLoRAMerged:", out_2)
print("Original layer      :", layer(x))
print("Equivalent:", torch.allclose(out_1, out_2, atol=1e-6))

LinearWithLoRA      : tensor([[-18.4803,   5.6285]], grad_fn=<AddBackward0>)
LinearWithLoRAMerged: tensor([[-18.4803,   5.6285]], grad_fn=<AddmmBackward0>)
Original layer      : tensor([[0.6639, 0.4487]], grad_fn=<AddmmBackward0>)
Equivalent: True


Both implementations give the **same output**, also when the LoRA matrices are non-zero, because

$$ x W^\top + b + \alpha\, x A B \;=\; x \left(W + \alpha (A B)^\top\right)^\top + b $$

- `LinearWithLoRA` keeps two separate branches (useful during training: the frozen `W` and the small `A`, `B` are clearly separated).
- `LinearWithLoRAMerged` folds the update into a single weight matrix. After training, we can compute $W' = W + \alpha (AB)^\top$ **once** and use a plain linear layer: inference then has **no extra cost** compared with the original model, which is a key advantage of LoRA over other adapters.

## 🌟 Exercise 5: Implementing a Multilayer Perceptron (MLP) and Replacing Layers with LoRA

In [9]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),   # layers[0]
            nn.ReLU(),                               # layers[1]
            nn.Linear(num_hidden_1, num_hidden_2),   # layers[2]
            nn.ReLU(),                               # layers[3]
            nn.Linear(num_hidden_2, num_classes),    # layers[4]
        )

    def forward(self, x):
        x = self.layers(x)
        return x

# Architecture
num_features = 784      # 28 x 28 MNIST images, flattened
num_hidden_1 = 128
num_hidden_2 = 256
num_classes = 10

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 0.005
num_epochs = 2

torch.manual_seed(random_seed)
model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(DEVICE)
print(model)
print(optimizer_pretrained)

cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)


### Loading dataset

In [10]:
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 64

# Note: transforms.ToTensor() scales input images to 0-1 range
train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)

test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


### Define evaluation

In [11]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.view(-1, 28 * 28).to(device)   # flatten the images
            targets = targets.to(device)
            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()
        return correct_pred.float() / num_examples * 100

### Training

In [12]:
import time

def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.view(-1, 28 * 28).to(device)
            targets = targets.to(device)

            # forward and back propagation
            logits = model(features)
            loss = F.cross_entropy(logits, targets)
            optimizer.zero_grad()

            loss.backward()

            # update model parameters
            optimizer.step()

            # logging
            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch + 1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch + 1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time) / 60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time) / 60))


train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch: 001/002|Batch 000/938| Loss: 2.3023


Epoch: 001/002|Batch 400/938| Loss: 0.1868


Epoch: 001/002|Batch 800/938| Loss: 0.1113


Epoch: 001/002 training accuracy: 96.82%
Time elapsed: 0.13 min
Epoch: 002/002|Batch 000/938| Loss: 0.0207


Epoch: 002/002|Batch 400/938| Loss: 0.0776


Epoch: 002/002|Batch 800/938| Loss: 0.0742


Epoch: 002/002 training accuracy: 97.64%
Time elapsed: 0.25 min
Total Training Time: 0.25 min


Test accuracy: 97.09%


### Replacing Linear with LoRA Layers

Using `LinearWithLoRAMerged`, we add the LoRA layers by replacing the original `Linear` layers of the multilayer perceptron:

In [13]:
import copy

model_lora = copy.deepcopy(model)

model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)
model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=256, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=256, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)


In [14]:
print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

Test accuracy orig model:97.09%


Test accuracy LoRA model:97.09%


The printed architecture shows that the three `Linear` layers are now wrapped in `LinearWithLoRAMerged` (each containing the original `linear` and a `lora` sub-module), while the `ReLU` layers are unchanged. Right after the replacement, the LoRA model has **exactly the same test accuracy** as the original model, because every `B` matrix is still zero.

## 🌟 Exercise 6: Freezing the Original Linear Layers and Training LoRA

We freeze the original `Linear` layers and only make the `LoRALayer` parameters trainable:

In [15]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            # recursively freeze linear layers in children modules
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}:{param.requires_grad}')

layers.0.linear.weight:False
layers.0.linear.bias:False
layers.0.lora.A:True
layers.0.lora.B:True
layers.2.linear.weight:False
layers.2.linear.bias:False
layers.2.lora.A:True
layers.2.lora.B:True
layers.4.linear.weight:False
layers.4.linear.bias:False
layers.4.lora.A:True
layers.4.lora.B:True


In [16]:
def count_parameters(m, trainable_only=False):
    return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

total = count_parameters(model_lora)
trainable = count_parameters(model_lora, trainable_only=True)
print(f"Original MLP parameters (all trainable): {count_parameters(model):,}")
print(f"LoRA model - total parameters: {total:,} | trainable (LoRA only): {trainable:,} "
      f"({trainable / count_parameters(model):.1%} of the original model)")
for i in [0, 2, 4]:
    l = model_lora.layers[i]
    print(f"  layers[{i}]: frozen W {tuple(l.linear.weight.shape)} = {l.linear.weight.numel():,} params | "
          f"LoRA A {tuple(l.lora.A.shape)} + B {tuple(l.lora.B.shape)} = {l.lora.A.numel() + l.lora.B.numel():,} params")

Original MLP parameters (all trainable): 136,074
LoRA model - total parameters: 142,322 | trainable (LoRA only): 6,248 (4.6% of the original model)
  layers[0]: frozen W (128, 784) = 100,352 params | LoRA A (784, 4) + B (4, 128) = 3,648 params
  layers[2]: frozen W (256, 128) = 32,768 params | LoRA A (128, 4) + B (4, 256) = 1,536 params
  layers[4]: frozen W (10, 256) = 2,560 params | LoRA A (256, 4) + B (4, 10) = 1,064 params


Based on the True and False values above, we can confirm that **only the LoRA layers are trainable** now (True means trainable, False means frozen): all `linear.weight` and `linear.bias` tensors are frozen, and only the `lora.A` / `lora.B` matrices will be updated. In practice we would then train the network with this LoRA configuration on a new dataset or task; here we continue training on MNIST to show that the frozen model can still be improved with very few parameters.

In [17]:
# Weights of a frozen layer before LoRA training (to check they do not change)
w0_before = model_lora.layers[0].linear.weight.detach().clone()

optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

Epoch: 001/002|Batch 000/938| Loss: 0.1188


Epoch: 001/002|Batch 400/938| Loss: 0.3309


Epoch: 001/002|Batch 800/938| Loss: 0.0103


Epoch: 001/002 training accuracy: 97.50%
Time elapsed: 0.12 min
Epoch: 002/002|Batch 000/938| Loss: 0.1016


Epoch: 002/002|Batch 400/938| Loss: 0.0237


Epoch: 002/002|Batch 800/938| Loss: 0.1536


Epoch: 002/002 training accuracy: 97.62%
Time elapsed: 0.25 min
Total Training Time: 0.25 min


Test accuracy LoRA finetune: 96.95%


In [18]:
print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print("\nFrozen weights unchanged after LoRA training:", torch.equal(w0_before, model_lora.layers[0].linear.weight))
print("LoRA B matrix is no longer zero:", bool(model_lora.layers[0].lora.B.abs().sum() > 0))

Test accuracy orig model:97.09%


Test accuracy LoRA model:96.95%

Frozen weights unchanged after LoRA training: True
LoRA B matrix is no longer zero: True


In [19]:
# Final step of a LoRA workflow: merge the learned update into plain Linear layers for inference
merged_model = copy.deepcopy(model)
with torch.no_grad():
    for i in [0, 2, 4]:
        l = model_lora.layers[i]
        merged_model.layers[i].weight.copy_(l.linear.weight + l.lora.alpha * (l.lora.A @ l.lora.B).T)
print(merged_model)
print(f"Merged plain-MLP test accuracy: {compute_accuracy(merged_model, test_loader, DEVICE):.2f}% "
      f"(same as the LoRA model, with no extra inference cost)")

MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


Merged plain-MLP test accuracy: 96.95% (same as the LoRA model, with no extra inference cost)


## Results and conclusion

| Model | Trainable parameters | Test accuracy |
|---|---|---|
| Original MLP (784 → 128 → 256 → 10), trained 2 epochs | 136,074 (100%) | 97.09% |
| Same MLP + LoRA (rank 4, alpha 8), **before** LoRA training | – | 97.09% (identical, because B = 0) |
| LoRA model after 2 epochs with **only the LoRA matrices trainable** | **6,248 (4.6%)** | 96.95% |
| LoRA update merged back into plain `Linear` layers | – | 96.95% (identical to the LoRA model) |

**What the experiments show:**
- `LoRALayer` computes the low-rank update `alpha · x A B`; because **B is initialised to zero**, wrapping a layer with `LinearWithLoRA` (Exercise 3) or `LinearWithLoRAMerged` leaves its output **unchanged**, and the two implementations give the same result even with non-zero LoRA matrices (Exercise 4).
- After `freeze_linear_layers`, only the `lora.A` and `lora.B` tensors have `requires_grad=True`: **6,248 trainable parameters instead of 136,074 (4.6%)**. For the first layer, the LoRA update needs 3,648 parameters instead of the 100,352 of the full weight matrix. With huge models (billions of parameters, matrices of 4096 × 4096), the saving is even bigger: rank 8 needs 65,536 parameters instead of 16.8 million per matrix (0.4%).
- After LoRA training, the **frozen weights are exactly unchanged** (checked with `torch.equal`) while the B matrices are no longer zero: all the learning went into the small adapters.
- The test accuracy after LoRA fine-tuning (96.95%) is essentially the same as the original model (97.09%; the small drop is within the noise of two more epochs with a fairly high learning rate of 0.005). This is expected: we fine-tuned **on the same task and the same data** the model was already trained on, so there was nothing new to learn, as noted in the template. The real benefit of LoRA appears when adapting a pretrained model to a **new task or domain** (e.g. a pretrained LLM to legal documents or a specific writing style): only the small adapters are trained and stored (a few MB per task instead of a full copy of the model), several adapters can share the same frozen base model, and training needs much less GPU memory because the optimizer states are only kept for the LoRA parameters.
- Finally, the learned update can be **merged** into the original weights (`W' = W + alpha (A B)^T`): the merged plain MLP has exactly the same accuracy as the LoRA model, with **no extra cost at inference**, which is a key advantage of LoRA over other adapter methods.